# HW7 RAG 1｜把「實驗室安全衛生工作守則」變成向量資料庫

**中興 4113052132 生科3 蔡宸維**　｜　主題：🧯 **實驗室保命 RAG：問阿哲學長之前，先查守則**

## 0. 我選的資料與動機


| 項目 | 內容 |
|---|---|
| 文件 | 《國立中興大學微生物暨公共衛生學研究所安全衛生工作守則》 |
| 版本 | 94 年訂定，104 年 3 月 2 日所務會議修訂名稱及全文通過 |
| 來源 | 該所網站公開 PDF：[1040302.pdf](https://www.gimph.nchu.edu.tw/storage/upload/download/1040302.pdf) |
| 篇幅 | 共 37 頁：第 1–16 頁中文版，第 16–37 頁英文譯本（譯本註明「以中文版為準」）|
| 內容 | 10 章 26 條：管理權責、設備檢查、實驗室守則、有機溶劑、高壓氣體、毒性氣體、放射性物質、急救、防護具、事故通報 |
| 我的處理 | 只取**中文版**，依「章／條／款」切成 **35 篇**（一節一檔 TXT）；只接回 PDF 換行，**不改寫條文**；每篇另加章節、類別、原文頁碼與一題檢索練習問題 |


**為什麼選這份？** 我是生科系學生，每學期都在實驗室，進實驗室前都要簽「已閱讀安全衛生守則」。但真的被藥品噴到、或想知道「藥品冰箱能不能冰便當」時，沒有人會去翻 37 頁的 PDF。把守則做成 RAG，就能用口語發問，直接找到對應條文。這也延續 HW6 的「實驗翻車急救站」：HW6 的阿哲學長只靠模型記憶回答，HW7 改成**必須引用守則條文**。

## 與老師範例的差異

| | 老師範例 | 我的作業 |
|---|---|---|
| 資料 | 大學生打工權益 16 篇教學摘要 | 中興大學系所安全衛生工作守則**原文** 35 節 |
| 讀資料 | 手動上傳 ZIP | ⭐ **從 GitHub 網址自動下載**（助教可直接執行），失敗才改手動上傳 |
| embedding 文字 | 標題＋正文 | 標題＋⭐**章節編號**＋正文（可用「第幾條」查詢）|
| metadata | 標題、網址、日期 | 再加 ⭐**章節、類別、原文頁碼** |
| 檢索檢查 | 試查一題 | ⭐ 用 35 題練習問題**自動評分 Hit@1／Hit@3**，並畫出類別分布 |
| 匯出 | `vector_embedding.zip` | `rag-data.zip`（依作業要求命名）|

> **操作方式：執行階段 → 全部執行**（建議在「變更執行階段類型」選 GPU，CPU 也能跑但較慢）。
> 本份**不呼叫 LLM**，也不需要 Groq 金鑰；第二份 RAG02 才會用 AIsuite 回答並以 Gradio 展示。
>
> 流程：原始文件 ZIP（網址）→ 擷取 RAG 文字與 metadata → EmbeddingGemma 2 → FAISS → 匯出 `rag-data.zip`

## 1. 安裝套件

與老師範例相同，固定 Sentence Transformers、Transformers 與 FAISS 的版本，方便重現。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 引入老師的固定套件與其他工具

第一格是老師課程固定引入的套件（NumPy、pandas、matplotlib）；本作業用 pandas 顯示表格、用 matplotlib 畫類別分布。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import importlib.metadata as package_metadata
import urllib.request
import hashlib
import io
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

## 3. 集中設定

- `EXPECTED_DOCUMENT_COUNT = 35`：守則切成 35 節。
- `EXPORT_ZIP_NAME = "rag-data.zip"`：依作業規定的檔名。
- ⭐ `DATA_URL`：原始文件 ZIP 放在我的 GitHub 公開 repo（[Alan47495/genai-hw7-lab-safety-rag](https://github.com/Alan47495/genai-hw7-lab-safety-rag)），**程式直接下載，助教不必手動上傳**。
- 前綴（prompt template）與老師範例相同，符合 EmbeddingGemma 2 的文件／查詢格式。

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = 35  # 守則切成 35 節
EXPORT_ZIP_NAME = "rag-data.zip"  # 作業指定檔名

# ⭐ 原始文件 ZIP 的公開下載網址（GitHub raw），助教可直接執行
DATA_URL = "https://raw.githubusercontent.com/Alan47495/genai-hw7-lab-safety-rag/main/lab-safety-docs.zip"

# text 本身已含標題與章節；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

## 4. ⭐ 從網址下載原始文件 ZIP（助教可直接執行）

老師範例用 `files.upload()` 手動上傳；這裡改成**用 `urllib` 從 GitHub 網址下載**。只有在網址失效或沒有網路時，才退回手動上傳。

解壓縮的安全檢查沿用老師範例：只取 TXT、略過隱藏檔與 `__MACOSX`、確認路徑不會跑出暫存資料夾、不允許重複檔名。

In [ ]:
def download_bytes(url, timeout=60):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


try:
    archive_bytes = download_bytes(DATA_URL)
    archive_name = Path(urlparse(DATA_URL).path).name
    print("已從網址下載：", DATA_URL)
except Exception as exc:
    print(f"網址下載失敗（{type(exc).__name__}），改為手動上傳原始文件 ZIP。")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("請一次上傳一個原始文件 ZIP 檔。")
    archive_name = zip_names[0]
    archive_bytes = uploaded[archive_name]

SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_documents_"))

# 僅解開 TXT，並確保 ZIP 中的路徑不會寫到暫存資料夾外。
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [
        member for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".txt")
        and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
    ]
    if not members:
        raise ValueError("ZIP 中找不到 TXT。")
    destinations = set()
    for member in members:
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError(f"ZIP 包含不適當的檔案路徑：{member.filename}")
        if target in destinations:
            raise ValueError(f"ZIP 有重複檔名：{member.filename}")
        destinations.add(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))

txt_paths = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)
print("文件 ZIP：", archive_name, f"（{len(archive_bytes):,} bytes）")
print("TXT 數量：", len(txt_paths))
for path in txt_paths:
    print("•", path.relative_to(DATA_DIR).as_posix())
if EXPECTED_DOCUMENT_COUNT is not None and len(txt_paths) != EXPECTED_DOCUMENT_COUNT:
    print(f"提醒：本次為 {len(txt_paths)} 篇，與預期的 {EXPECTED_DOCUMENT_COUNT} 篇不同；仍可繼續。")

## 5. 先看一篇原始文件

每篇 TXT 是我依守則章節切出來的，格式參考老師的教材：

- **標題／章節／類別／原文頁碼**：方便回答時標示出處
- **【內容】**：守則條文原文（只接回 PDF 換行）
- **【檢索練習問題】**：我替每一節寫的一題口語問題，**不放進 embedding**，留到第 13 節做檢索評分

In [ ]:
SAMPLE_INDEX = 11  # 12_實驗室守則_儲存與排氣.txt；可改成 0～34 看其他篇
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

## 6. ⭐ 修改後的 `extract_rag_text()`：標題＋章節＋條文

和老師範例相比，我把**章節編號**接在標題後面。這樣使用者問「第十二條講什麼？」「急救那一章」時，章節文字也能幫忙檢索。

【檢索練習問題】與【提醒】不會混入；找不到【內容】時明確報錯，不把整份文件拿去 embedding。

In [ ]:
def extract_rag_text(raw_text):
    """取出「標題（章節）＋【內容】條文」；練習問題與提醒不放進 embedding。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    title_match = re.search(r"^標題[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    section_match = re.search(r"^章節[：:]\s*(.+)$", normalized, flags=re.MULTILINE)
    content_match = re.search(
        r"^【內容】[ \t]*\n(?P<body>.*?)(?=^【[^】\n]+】|\Z)",
        normalized,
        flags=re.MULTILINE | re.DOTALL,
    )
    if content_match is None or not content_match.group("body").strip():
        raise ValueError("找不到非空的【內容】正文；請修改 extract_rag_text() 以配合文件格式。")
    title = title_match.group(1).strip() if title_match else ""
    section = section_match.group(1).strip() if section_match else ""
    heading = f"{title}（{section}）" if title and section else (title or section)
    body = content_match.group("body").strip()
    return f"{heading}\n\n{body}" if heading else body


def extract_practice_question(raw_text):
    """取出【檢索練習問題】，只用於第 13 節的檢索評分，不放進知識庫。"""
    match = re.search(r"^【檢索練習問題】[ \t]*\n(?P<q>.+?)$", raw_text, flags=re.MULTILINE)
    return match.group("q").strip() if match else None


sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))
print("練習問題（不進 embedding）：", extract_practice_question(sample_raw))

## 7. ⭐ 修改後的 `extract_metadata()`：加入章節、類別、頁碼

metadata 用於追溯出處與介面顯示，不整包送進 embedding。新增的欄位：

| 欄位 | 用途 |
|---|---|
| `section` | 第幾章第幾條，回答時標明條文位置 |
| `category` | 化學品／急救／氣體與壓力設備／設備與儀器／一般守則／管理與通報，RAG02 可用來篩選 |
| `pages` | 原文 PDF 頁碼，方便回去核對 |
| `source_name` | 守則全名與修訂日期 |

找不到的欄位用 `None`，不自行捏造。

In [ ]:
def extract_metadata(raw_text, filename):
    """取出守則每一節的描述資訊。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    def read_field(name):
        match = re.search(rf"^{re.escape(name)}[：:][ \t]*(.*)$", normalized, flags=re.MULTILINE)
        return match.group(1).strip() or None if match else None

    return {
        "title": read_field("標題") or Path(filename).stem,
        "section": read_field("章節"),
        "category": read_field("類別") or "未分類",
        "pages": read_field("原文頁碼"),
        "source_name": read_field("來源文件"),
        "source_url": read_field("來源網址"),
        "date": read_field("教材整理日期"),
        "filename": filename,
        "document_type": read_field("資料類型") or "未標示",
    }

sample_metadata = extract_metadata(sample_raw, sample_path.relative_to(DATA_DIR).as_posix())
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))

## 8. 批次整理成統一格式

每篇得到 `doc_id`、`text`、`metadata`；練習問題另外存在 `practice_questions`，不進知識庫。缺少章節、頁碼或來源時會提醒。

In [ ]:
documents = []
practice_questions = {}
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")
        rag_text = extract_rag_text(raw_text)
        metadata = extract_metadata(raw_text, filename)
    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc
    doc_id = f"doc_{number:03d}"
    documents.append({"doc_id": doc_id, "text": rag_text, "metadata": metadata})
    practice_questions[doc_id] = extract_practice_question(raw_text)

if not documents:
    raise ValueError("沒有可建立索引的文件。")
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"], "章節": doc["metadata"]["section"],
     "類別": doc["metadata"]["category"], "頁碼": doc["metadata"]["pages"],
     "字元數": len(doc["text"])}
    for doc in documents
])
display(summary)
for doc in documents:
    missing = [key for key in ("section", "pages", "source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))
print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

### ⭐ 8-1. 各類別篇數（用老師固定套件的 matplotlib 畫圖）

Colab 預設字型沒有中文，所以圖上的類別用英文標示，對照表印在下方。

In [ ]:
CATEGORY_EN = {
    "化學品": "Chemicals", "急救": "First aid", "管理與通報": "Management & reporting",
    "一般守則": "General rules", "氣體與壓力設備": "Gas & pressure", "設備與儀器": "Equipment",
}
category_counts = summary["類別"].value_counts()
fig, ax = plt.subplots(figsize=(7, 3.2))
labels = [CATEGORY_EN.get(c, c) for c in category_counts.index][::-1]
ax.barh(labels, category_counts.values[::-1], color="#2a9d8f")
for y, value in enumerate(category_counts.values[::-1]):
    ax.text(value + 0.1, y, str(value), va="center")
ax.set_xlabel("Number of sections")
ax.set_title("Lab safety code: sections per category")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"類別": category_counts.index,
                      "English": [CATEGORY_EN.get(c, c) for c in category_counts.index],
                      "篇數": category_counts.values}))

## 9. 載入 EmbeddingGemma 2

與老師範例相同：先查模型確切版本（commit）再載入，並把版本寫進 `config.json`，讓 RAG02 載入同一版本。首次下載約 1.5 GB，需要等待。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())

## 10. 檢查輸入長度，避免靜默截斷

守則每節約 150～650 字，加上前綴後都遠低於 2048 tokens，所以採「一節一向量」不必再切塊。超過上限會直接停止，不偷偷截斷。

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "標題": summary["標題"], "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("token 上限：", TOKEN_LIMIT, "／最長一篇：", max(token_counts), "tokens")
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])

## 11. 產生文件向量並建立 FAISS 精確搜尋索引

向量正規化後，內積就是 cosine similarity；`IndexFlatIP` 會和每一篇比較，保證找到分數最高者。FAISS 回傳的是向量編號，要透過 `index_doc_ids` 找回條文。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")

index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("索引類型：", type(index).__name__, "／索引向量數：", index.ntotal, "／維度：", index.d)

## 12. 定義問題向量化與搜尋函式，試查幾題

問題用同一個模型、但用查詢專用前綴。先查一題「有答案」的，再查一題守則**沒有**的——FAISS 仍會回傳最相近的條文，**相似度不是正確率**。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results


def show_results(question, top_k=3):
    results = search_documents(question, top_k)
    print("問題：", question)
    display(pd.DataFrame([
        {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
         "章節": r["metadata"]["section"], "cosine_similarity": round(r["score"], 4)}
        for r in results
    ]))
    return results

In [ ]:
QUESTION = "化學藥品噴到眼睛要沖多久？"
TOP_K = 3
results = show_results(QUESTION, TOP_K)
print(f"\n[{results[0]['doc_id']}] {results[0]['metadata']['title']}（{results[0]['metadata']['pages']}）")
print(results[0]["text"])

In [ ]:
# 守則裡沒有答案的問題：仍然會有「最相近」的條文
_ = show_results("中興大學學餐哪一家最好吃？", 3)

## 13. ⭐ 檢索評分：35 題練習問題能不能找回自己那一節？

每一節我都寫了一題口語的【檢索練習問題】（例如「放藥品的冰箱可以冰我的便當嗎？」），而且**沒有放進 embedding**。這裡把 35 題逐一拿去搜尋：

- **Hit@1**：第 1 名就是出題的那一節
- **Hit@3**：出題的那一節在前 3 名內

這能看出「口語問題 vs. 法規條文」之間的檢索效果，也幫我決定 RAG02 預設取回 3 篇。

In [ ]:
eval_rows = []
for doc in documents:
    question = practice_questions.get(doc["doc_id"])
    if not question:
        continue
    hits = search_documents(question, 3)
    ranked_ids = [h["doc_id"] for h in hits]
    eval_rows.append({
        "doc_id": doc["doc_id"], "標題": doc["metadata"]["title"], "練習問題": question,
        "第1名": ranked_ids[0], "第1名分數": round(hits[0]["score"], 4),
        "Hit@1": ranked_ids[0] == doc["doc_id"], "Hit@3": doc["doc_id"] in ranked_ids,
    })
eval_table = pd.DataFrame(eval_rows)
print(f"題數：{len(eval_table)}")
print(f"Hit@1：{eval_table['Hit@1'].mean():.1%}")
print(f"Hit@3：{eval_table['Hit@3'].mean():.1%}")
print("\n沒有進入前 3 名的題目：")
display(eval_table.loc[~eval_table["Hit@3"], ["doc_id", "標題", "練習問題", "第1名"]])

In [ ]:
# 依類別看檢索表現
eval_table["類別"] = [documents_by_id[d]["metadata"]["category"] for d in eval_table["doc_id"]]
display(eval_table.groupby("類別")[["Hit@1", "Hit@3"]].mean().round(2))

## 14. 保存索引、文件、編號對照與設定

與老師範例相同的四個檔案；`config.json` 另外記錄守則來源與類別清單，RAG02 會用類別做篩選。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_section_one_vector",
    "document_text_includes_title": True,
    "source_document": documents[0]["metadata"]["source_name"],
    "categories": sorted({doc["metadata"]["category"] for doc in documents}),
    "retrieval_eval": {
        "questions": int(len(eval_table)),
        "hit_at_1": round(float(eval_table["Hit@1"].mean()), 4),
        "hit_at_3": round(float(eval_table["Hit@3"].mean()), 4),
    },
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")

## 15. 重新載入，驗證 RAG02 所需的資料

沿用老師範例的檢查：檔案雜湊、文件對照、索引維度、保存前後同一個問題向量的排名與分數一致，以及 FAISS 與 NumPy 全矩陣內積的結果一致。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

## 16. 打包成 `rag-data.zip` 並下載

ZIP 根目錄包含四個檔案（不含模型權重與任何金鑰）。

我已把同樣流程產生的 `rag-data.zip` 上傳到 GitHub：`https://raw.githubusercontent.com/Alan47495/genai-hw7-lab-safety-rag/main/rag-data.zip`，RAG02 會直接從這個網址下載，所以助教**不需要先跑這一份**。若更換文件或修改擷取函式，要重新匯出並更新 GitHub 上的檔案，不要混用舊索引與新文件。

In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
print("路徑：", EXPORT_ZIP)
try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("（非 Colab 環境，請直接到上面的路徑取檔。）")

## 17. 小結

1. **資料**：中興大學微生物暨公共衛生學研究所安全衛生工作守則（中文版），依章／條／款切成 35 節，一節一向量。
2. **我修改的地方**：從網址下載 ZIP、`extract_rag_text()` 加入章節、`extract_metadata()` 加入章節／類別／頁碼、類別分布圖、35 題 Hit@1／Hit@3 檢索評分、匯出為 `rag-data.zip`。
3. **觀察**（我在本機 CPU 跑的結果，Colab GPU 精度不同，數字可能略有差異）：
   - 35 題練習問題的 **Hit@1 約 74%、Hit@3 約 83%**。口語問題（如「冰便當」）和條文（「冷藏化學藥品之冰箱不得放置食品」）用字差很多，語意向量仍能找到；所以 RAG02 預設取回 3 篇。
   - 化學品類 6 題全中；最弱的是「設備與儀器」與「一般守則」，因為條文很抽象（例如「防止機械、器具、設備等引起之危害」），和具體的口語問題距離較遠。
   - 沒找到的題目，第 1 名常常是 doc_013（防護具、化學品濺潑），它像一塊「磁鐵」，只要問題帶有「安全、受傷、怎麼辦」的語氣就容易被拉過去。
   - 連「八小時內通報」這種帶關鍵數字的問題也可能失手，表示 embedding 是比語意、不是比關鍵字；守則沒有的問題（學餐）分數明顯較低（約 0.60），但仍會回傳結果。
4. **下一步**：RAG02 從 GitHub 下載 `rag-data.zip`，讀取 Colab 密鑰 `Groq`，用 AIsuite 呼叫 `groq:openai/gpt-oss-120b`，讓「阿哲學長」只根據條文回答，並用 Gradio 展示。